<a href="https://colab.research.google.com/github/Raoina/Spectra-2-Image/blob/main/notebooks/Models/1D_CNN_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ====== Imports ======
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import iqr

In [2]:
if torch.cuda.is_available():
    print("CUDA is available! Running on GPU.")
else:
    print("CUDA is not available. Running on CPU.")

CUDA is available! Running on GPU.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
file_path = '/content/drive/MyDrive/LUCAS.SOIL_corr.csv'

In [ ]:
df = pd.read_csv(file_path)

/tmp/ipython-input-2618208116.py:1: DtypeWarning: Columns (1,2,4216,4231,4234,4237,4274) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


In [ ]:
# df.head()

/tmp/ipython-input-1058064533.py:1: DtypeWarning: Columns (1,2,4216,4231,4234,4237,4274) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


,Unnamed: 0,sample.ID,ID,date,spc.400,spc.400.5,spc.401,spc.401.5,spc.402,spc.402.5,...,WRBLV1,WRBADJ1,PARMADO1,PARMADO2,PARMADO3,PARMASE1,PARMASE2,PARMASE3,MIN_TOP,mineral
0,4,100,100,2011-01-24 16:02:25,0.831705,0.839111,0.846473,0.853773,0.860988,0.868098,...,CM,dy,3.0,31.0,310.0,5.0,56.0,561.0,KQ,mineral
1,5,1000,1000,2010-09-30 14:50:01,0.706027,0.714901,0.723727,0.732480,0.741142,0.749684,...,CM,dy,3.0,31.0,311.0,3.0,31.0,310.0,KQ,mineral
2,6,10000,10000,2010-10-19 12:06:16,0.666238,0.676472,0.686654,0.696757,0.706753,0.716615,...,CM,ca,2.0,21.0,214.0,0.0,0.0,0.0,MS,mineral
3,7,10001,10001,2010-10-19 15:00:01,0.698136,0.706548,0.714909,0.723197,0.731384,0.739448,...,CM,dy,3.0,31.0,310.0,4.0,42.0,420.0,KQ,mineral
4,8,10002,10002,2011-04-30 12:26:27,0.675433,0.684820,0.694163,0.703438,0.712620,0.721693,...,LV,ha,7.0,71.0,710.0,6.0,62.0,620.0,M,mineral


In [ ]:
# print(df.columns.tolist())

['Unnamed: 0', 'sample.ID', 'ID', 'date', 'spc.400', 'spc.400.5', 'spc.401', 'spc.401.5', 'spc.402', 'spc.402.5', 'spc.403', 'spc.403.5', 'spc.404', 'spc.404.5', 'spc.405', 'spc.405.5', 'spc.406', 'spc.406.5', 'spc.407', 'spc.407.5', 'spc.408', 'spc.408.5', 'spc.409', 'spc.409.5', 'spc.410', 'spc.410.5', 'spc.411', 'spc.411.5', 'spc.412', 'spc.412.5', 'spc.413', 'spc.413.5', 'spc.414', 'spc.414.5', 'spc.415', 'spc.415.5', 'spc.416', 'spc.416.5', 'spc.417', 'spc.417.5', 'spc.418', 'spc.418.5', 'spc.419', 'spc.419.5', 'spc.420', 'spc.420.5', 'spc.421', 'spc.421.5', 'spc.422', 'spc.422.5', 'spc.423', 'spc.423.5', 'spc.424', 'spc.424.5', 'spc.425', 'spc.425.5', 'spc.426', 'spc.426.5', 'spc.427', 'spc.427.5', 'spc.428', 'spc.428.5', 'spc.429', 'spc.429.5', 'spc.430', 'spc.430.5', 'spc.431', 'spc.431.5', 'spc.432', 'spc.432.5', 'spc.433', 'spc.433.5', 'spc.434', 'spc.434.5', 'spc.435', 'spc.435.5', 'spc.436', 'spc.436.5', 'spc.437', 'spc.437.5', 'spc.438', 'spc.438.5', 'spc.439', 'spc.439.5'

In [ ]:
spc_columns = [col for col in df.columns if col.startswith('spc')]
spc_df = df[spc_columns]

In [ ]:
# display(spc_df.head())

,spc.400,spc.400.5,spc.401,spc.401.5,spc.402,spc.402.5,spc.403,spc.403.5,spc.404,spc.404.5,...,spc.2495,spc.2495.5,spc.2496,spc.2496.5,spc.2497,spc.2497.5,spc.2498,spc.2498.5,spc.2499,spc.2499.5
0,0.831705,0.839111,0.846473,0.853773,0.860988,0.868098,0.875088,0.881935,0.888626,0.895142,...,0.552283,0.552303,0.552319,0.552333,0.552342,0.552346,0.552345,0.552338,0.552329,0.552314
1,0.706027,0.714901,0.723727,0.732480,0.741142,0.749684,0.758085,0.766323,0.774371,0.782215,...,0.424424,0.424536,0.424642,0.424742,0.424837,0.424924,0.425009,0.425088,0.425165,0.425256
2,0.666238,0.676472,0.686654,0.696757,0.706753,0.716615,0.726321,0.735840,0.745154,0.754237,...,0.426412,0.426590,0.426763,0.426931,0.427094,0.427254,0.427409,0.427561,0.427712,0.427900
3,0.698136,0.706548,0.714909,0.723197,0.731384,0.739448,0.747365,0.755111,0.762668,0.770012,...,0.593780,0.594077,0.594368,0.594653,0.594935,0.595211,0.595482,0.595749,0.596013,0.596341
4,0.675433,0.684820,0.694163,0.703438,0.712620,0.721693,0.730629,0.739405,0.748003,0.756403,...,0.401095,0.401195,0.401289,0.401380,0.401467,0.401554,0.401637,0.401720,0.401801,0.401902


In [ ]:
concen_columns = ['OC', 'N', 'CEC', 'pH.in.H2O', 'sand', 'clay']
concen_df = df[concen_columns]

In [ ]:
# display(concen_df.head())

,OC,N,CEC,pH.in.H2O,sand,clay
0,91.1,5.3,7.2,4.00,48.0,7.0
1,21.4,2.1,13.0,6.53,60.0,13.0
2,15.6,1.4,24.6,7.14,8.0,40.0
3,19.8,1.6,20.6,4.83,56.0,26.0
4,33.5,2.6,15.0,5.74,37.0,22.0


In [ ]:
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)

# print(df.isna().sum())

Unnamed: 0               0
sample.ID                0
ID                       0
date                     0
spc.400                  0
spc.400.5                0
spc.401                  0
spc.401.5                0
spc.402                  0
spc.402.5                0
spc.403                  0
spc.403.5                0
spc.404                  0
spc.404.5                0
spc.405                  0
spc.405.5                0
spc.406                  0
spc.406.5                0
spc.407                  0
spc.407.5                0
spc.408                  0
spc.408.5                0
spc.409                  0
spc.409.5                0
spc.410                  0
spc.410.5                0
spc.411                  0
spc.411.5                0
spc.412                  0
spc.412.5                0
spc.413                  0
spc.413.5                0
spc.414                  0
spc.414.5                0
spc.415                  0
spc.415.5                0
spc.416                  0
s

In [ ]:
# print(concen_df.isnull().sum())

OC              0
N               0
CEC             0
pH.in.H2O       0
sand         1097
clay         1097
dtype: int64


In [ ]:
# spc_df.to_csv("spc_data.csv", index=False)

In [ ]:
# concen_df.to_csv("concen_df.csv", index=False)

In [ ]:
class CNNWrapper(nn.Module):
    def __init__(self, mode="1d", input_length=4200, img_size=65, num_classes=1):
        super(CNNWrapper, self).__init__()
        self.mode = mode

        if mode == "1d":
            # 1D layers
            self.conv1 = nn.Conv1d(1, 64, kernel_size=3, padding=1)
            self.pool1d = nn.MaxPool1d(2)
            self.conv2 = nn.Conv1d(64, 128, 3, padding=1)
            self.pool1d = nn.MaxPool1d(2)
            self.conv3 = nn.Conv1d(128, 256, 3, padding=1)
            self.conv4 = nn.Conv1d(256, 256, 3, padding=1)
            self.pool1d = nn.MaxPool1d(2)
            self.conv5 = nn.Conv1d(256, 512, 3, padding=1)
            self.conv6 = nn.Conv1d(512, 512, 3, padding=1)
            self.pool1d = nn.MaxPool1d(2)
            self.conv7 = nn.Conv1d(512, 512, 3, padding=1)
            self.conv8 = nn.Conv1d(512, 512, 3, padding=1)
            self.pool1d = nn.MaxPool1d(2)

            # flatten size: input_length // (2^5) after 5 poolings
            flatten_dim = (input_length // (2**5)) * 512
            self.fc = nn.Linear(flatten_dim, num_classes)

        elif mode == "2d":
            # 2D layers
            self.conv1 = nn.Conv2d(1, 64, kernel_size=3, padding=1)
            self.pool2d = nn.MaxPool2d(2, 2)
            self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
            self.pool2d = nn.MaxPool2d(2, 2)
            self.conv3 = nn.Conv2d(128, 256, 3, padding=1)
            self.conv4 = nn.Conv2d(256, 256, 3, padding=1)
            self.pool2d = nn.MaxPool2d(2, 2)
            self.conv5 = nn.Conv2d(256, 512, 3, padding=1)
            self.conv6 = nn.Conv2d(512, 512, 3, padding=1)
            self.pool2d = nn.MaxPool2d(2, 2)
            self.conv7 = nn.Conv2d(512, 512, 3, padding=1)
            self.conv8 = nn.Conv2d(512, 512, 3, padding=1)
            self.pool2d = nn.MaxPool2d(2, 2)

            # flatten size: (65x65 → ~2x2 after 5 poolings)
            flatten_dim = 512 * 2 * 2
            self.fc = nn.Linear(flatten_dim, num_classes)

        else:
            raise ValueError("mode must be '1d' or '2d'")

    def forward(self, x):
        if self.mode == "1d":
            x = F.relu(self.conv1(x)); x = self.pool1d(x)
            x = F.relu(self.conv2(x)); x = self.pool1d(x)
            x = F.relu(self.conv3(x)); x = F.relu(self.conv4(x)); x = self.pool1d(x)
            x = F.relu(self.conv5(x)); x = F.relu(self.conv6(x)); x = self.pool1d(x)
            x = F.relu(self.conv7(x)); x = F.relu(self.conv8(x)); x = self.pool1d(x)
            x = x.view(x.size(0), -1)
            return self.fc(x)

        elif self.mode == "2d":
            x = F.relu(self.conv1(x)); x = self.pool2d(x)
            x = F.relu(self.conv2(x)); x = self.pool2d(x)
            x = F.relu(self.conv3(x)); x = F.relu(self.conv4(x)); x = self.pool2d(x)
            x = F.relu(self.conv5(x)); x = F.relu(self.conv6(x)); x = self.pool2d(x)
            x = F.relu(self.conv7(x)); x = F.relu(self.conv8(x)); x = self.pool2d(x)
            x = x.view(x.size(0), -1)
            return self.fc(x)

In [4]:
# ====== Device ======
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [5]:
# ==================== Load Data ====================
file_path = '/content/drive/MyDrive/LUCAS.SOIL_corr.csv'
df = pd.read_csv(file_path)
print("df shape:", df.shape)

/tmp/ipython-input-961292355.py:3: DtypeWarning: Columns (1,2,4216,4231,4234,4237,4274) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


df shape: (19036, 4288)


In [6]:
# ======Free NaN ======
df_clean = df.dropna(subset=['sand', 'clay'])
print("Free NaN:", df_clean.shape)
# Spectral data and Target properties
spc_columns = [col for col in df_clean.columns if col.startswith('spc')]
concen_columns = ['OC', 'N', 'CEC', 'pH.in.H2O', 'sand', 'clay']

X_df = df_clean[spc_columns]
y_df = df_clean[concen_columns]

print("Free NaN:", df_clean.shape)
print("X shape:", X_df.shape)
print("y shape:", y_df.shape)

Free NaN: (17939, 4288)
Free NaN: (17939, 4288)
X shape: (17939, 4200)
y shape: (17939, 6)


In [7]:
df_subset = df_clean.head(4000)
print("Subset shape:", df_subset.shape)

X_df = df_subset[spc_columns]
y_df = df_subset[concen_columns]

print("X shape (subset):", X_df.shape)
print("y shape (subset):", y_df.shape)

Subset shape: (4000, 4288)
X shape (subset): (4000, 4200)
y shape (subset): (4000, 6)


In [8]:
# ====== Scaling ======
scaler_X = MinMaxScaler()
X_scaled = scaler_X.fit_transform(X_df.values)
X = torch.tensor(X_scaled, dtype=torch.float32).unsqueeze(1)  # [N,1,L]

scaler_y = MinMaxScaler()
y_scaled = scaler_y.fit_transform(y_df.values)
y = torch.tensor(y_scaled, dtype=torch.float32)  # [N, num_targets]

# ====== Check input length ======
input_length = X.shape[2]
print("Input length:", input_length)  # 4200

Input length: 4200


In [15]:
# ====== Hyperparameters ======
batch_size = 128
epochs = 100
learning_rate = 0.001
n_splits = 5

In [16]:
class CNNWrapper(nn.Module):
    def __init__(self, mode="1d", input_length=4200, num_classes=1):
        super(CNNWrapper, self).__init__()
        self.mode = mode

        if mode == "1d":
            self.conv1 = nn.Conv1d(1, 64, 3, padding=1)
            self.bn1 = nn.BatchNorm1d(64)
            self.pool = nn.MaxPool1d(2)

            self.conv2 = nn.Conv1d(64, 128, 3, padding=1)
            self.bn2 = nn.BatchNorm1d(128)

            self.conv3 = nn.Conv1d(128, 256, 3, padding=1)
            self.bn3 = nn.BatchNorm1d(256)
            self.conv4 = nn.Conv1d(256, 256, 3, padding=1)
            self.bn4 = nn.BatchNorm1d(256)

            self.conv5 = nn.Conv1d(256, 512, 3, padding=1)
            self.bn5 = nn.BatchNorm1d(512)
            self.conv6 = nn.Conv1d(512, 512, 3, padding=1)
            self.bn6 = nn.BatchNorm1d(512)

            self.conv7 = nn.Conv1d(512, 512, 3, padding=1)
            self.bn7 = nn.BatchNorm1d(512)
            self.conv8 = nn.Conv1d(512, 512, 3, padding=1)
            self.bn8 = nn.BatchNorm1d(512)

            flatten_dim = (input_length // (2**5)) * 512
            self.fc = nn.Linear(flatten_dim, num_classes)
        else:
            raise ValueError("Only 1D CNN implemented as in paper")

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x))); x = self.pool(x)
        x = F.relu(self.bn2(self.conv2(x))); x = self.pool(x)
        x = F.relu(self.bn3(self.conv3(x))); x = F.relu(self.bn4(self.conv4(x))); x = self.pool(x)
        x = F.relu(self.bn5(self.conv5(x))); x = F.relu(self.bn6(self.conv6(x))); x = self.pool(x)
        x = F.relu(self.bn7(self.conv7(x))); x = F.relu(self.bn8(self.conv8(x))); x = self.pool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

In [9]:
# @title
# batch_size = 32
# epochs = 80
# learning_rate = 0.001
# n_splits = 4

In [11]:
# @title
# class CNNWrapper(nn.Module):
#     def __init__(self, mode="1d", input_length=4200, num_classes=1):
#         super(CNNWrapper, self).__init__()
#         self.mode = mode

#         if mode == "1d":
#             self.conv1 = nn.Conv1d(1, 64, 3, padding=1)
#             self.bn1 = nn.BatchNorm1d(64)
#             self.pool = nn.MaxPool1d(2)

#             self.conv2 = nn.Conv1d(64, 32, 3, padding=1)
#             self.bn2 = nn.BatchNorm1d(32)

#             self.conv3 = nn.Conv1d(32, 256, 3, padding=1)
#             self.bn3 = nn.BatchNorm1d(256)
#             self.conv4 = nn.Conv1d(256, 256, 3, padding=1)
#             self.bn4 = nn.BatchNorm1d(256)

#             self.conv5 = nn.Conv1d(256, 512, 3, padding=1)
#             self.bn5 = nn.BatchNorm1d(512)
#             self.conv6 = nn.Conv1d(512, 512, 3, padding=1)
#             self.bn6 = nn.BatchNorm1d(512)

#             self.conv7 = nn.Conv1d(512, 512, 3, padding=1)
#             self.bn7 = nn.BatchNorm1d(512)
#             self.conv8 = nn.Conv1d(512, 512, 3, padding=1)
#             self.bn8 = nn.BatchNorm1d(512)

#             flatten_dim = (input_length // (2**5)) * 512
#             self.fc = nn.Linear(flatten_dim, num_classes)
#         else:
#             raise ValueError("Only 1D CNN implemented as in paper")

#     def forward(self, x):
#         x = F.relu(self.bn1(self.conv1(x))); x = self.pool(x)
#         x = F.relu(self.bn2(self.conv2(x))); x = self.pool(x)
#         x = F.relu(self.bn3(self.conv3(x))); x = F.relu(self.bn4(self.conv4(x))); x = self.pool(x)
#         x = F.relu(self.bn5(self.conv5(x))); x = F.relu(self.bn6(self.conv6(x))); x = self.pool(x)
#         x = F.relu(self.bn7(self.conv7(x))); x = F.relu(self.bn8(self.conv8(x))); x = self.pool(x)
#         x = x.view(x.size(0), -1)
#         return self.fc(x)

In [17]:
# ====== Train/Test Split ======
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)
print("Training+CV samples:", X_train_val.shape[0])
print("Test samples:", X_test.shape[0])

Training+CV samples: 2800
Test samples: 1200


In [18]:
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_val)):
    print(f"\n=== Fold {fold+1} Training ===")

    X_train, X_val = X_train_val[train_idx].to(device), X_train_val[val_idx].to(device)
    y_train, y_val = y_train_val[train_idx].to(device), y_train_val[val_idx].to(device)

    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    model = CNNWrapper(mode="1d", input_length=input_length, num_classes=y.shape[1]).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # -------- Training --------
    for epoch in range(epochs):
        model.train()
        running_loss = 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            # Add gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_loss += loss.item()
        avg_loss = running_loss / len(train_loader)
        if (epoch + 1) % 10 == 0:  # Print loss every 10 epochs
            print(f"Fold {fold+1} - Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.6f}")

    # -------- Validation --------
    model.eval()
    with torch.no_grad():
        y_pred = model(X_val).cpu().numpy()
        y_true = y_val.cpu().numpy()

    fold_rmse, fold_r2, fold_rpiq = [], [], []
    print(f"\n=== Fold {fold+1} Validation Metrics ===")
    for i, col in enumerate(concen_columns):
        rmse = np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))
        r2 = r2_score(y_true[:, i], y_pred[:, i])
        rpiq = iqr(y_true[:, i]) / rmse
        fold_rmse.append(rmse)
        fold_r2.append(r2)
        fold_rpiq.append(rpiq)
        print(f"{col}: RMSE={rmse:.4f}, R²={r2:.4f}, RPIQ={rpiq:.4f}")

    fold_results.append((fold_rmse, fold_r2, fold_rpiq))


=== Fold 1 Training ===
Fold 1 - Epoch 10/100, Loss: 0.079684
Fold 1 - Epoch 20/100, Loss: 0.029707
Fold 1 - Epoch 30/100, Loss: 0.021794
Fold 1 - Epoch 40/100, Loss: 0.019417
Fold 1 - Epoch 50/100, Loss: 0.014214
Fold 1 - Epoch 60/100, Loss: 0.010803
Fold 1 - Epoch 70/100, Loss: 0.010081
Fold 1 - Epoch 80/100, Loss: 0.007933
Fold 1 - Epoch 90/100, Loss: 0.006466
Fold 1 - Epoch 100/100, Loss: 0.005387

=== Fold 1 Validation Metrics ===
OC: RMSE=0.1558, R²=-0.6066, RPIQ=0.5218
N: RMSE=0.1131, R²=-0.2000, RPIQ=0.7096
CEC: RMSE=0.1409, R²=-0.2558, RPIQ=1.2118
pH.in.H2O: RMSE=0.1852, R²=-0.3322, RPIQ=1.3762
sand: RMSE=0.2873, R²=-0.4617, RPIQ=1.2327
clay: RMSE=0.1748, R²=-0.1877, RPIQ=1.1735

=== Fold 2 Training ===
Fold 2 - Epoch 10/100, Loss: 0.108679
Fold 2 - Epoch 20/100, Loss: 0.031365
Fold 2 - Epoch 30/100, Loss: 0.024263
Fold 2 - Epoch 40/100, Loss: 0.020509
Fold 2 - Epoch 50/100, Loss: 0.019051
Fold 2 - Epoch 60/100, Loss: 0.018171
Fold 2 - Epoch 70/100, Loss: 0.016664
Fold 2 - Ep

In [19]:
# ====== Average CV Metrics ======
avg_rmse = np.mean([r[0] for r in fold_results], axis=0)
avg_r2 = np.mean([r[1] for r in fold_results], axis=0)
avg_rpiq = np.mean([r[2] for r in fold_results], axis=0)

# Create DataFrame for average CV metrics
avg_cv_metrics = []
for i, col in enumerate(concen_columns):
    avg_cv_metrics.append({'Property': col, 'Avg RMSE': avg_rmse[i], 'Avg R²': avg_r2[i], 'Avg RPIQ': avg_rpiq[i]})
avg_cv_metrics_df = pd.DataFrame(avg_cv_metrics)

print("\n=== Average CV Metrics ===")
display(avg_cv_metrics_df)

# ====== Evaluate on Test Set ======
model.eval()
with torch.no_grad():
    y_test_pred = model(X_test.to(device)).cpu().numpy()
    y_test_true = y_test.numpy()

print("\n=== Test Set Metrics ===")
test_metrics = []
for i, col in enumerate(concen_columns):
    rmse = np.sqrt(mean_squared_error(y_test_true[:, i], y_test_pred[:, i]))
    r2 = r2_score(y_test_true[:, i], y_test_pred[:, i])
    rpiq = iqr(y_test_true[:, i]) / rmse
    test_metrics.append({'Property': col, 'RMSE': rmse, 'R²': r2})

test_metrics_df = pd.DataFrame(test_metrics)
display(test_metrics_df)


=== Average CV Metrics ===


,Property,Avg RMSE,Avg R²,Avg RPIQ
0,OC,0.098396,0.180196,0.895755
1,N,0.080914,0.244150,0.946938
2,CEC,0.103165,0.293215,1.637457
3,pH.in.H2O,0.177929,-0.369210,1.585939
4,sand,0.225835,0.133817,1.726470
5,clay,0.149302,0.158525,1.505874



=== Test Set Metrics ===


,Property,RMSE,R²
0,OC,0.099130,0.392282
1,N,0.087647,0.303640
2,CEC,0.099272,0.430766
3,pH.in.H2O,0.216494,-0.788890
4,sand,0.218760,0.257852
5,clay,0.182213,-0.109454


In [20]:
print(test_metrics_df['R²'].mean())

0.08103276292483012
